In [1]:
import nico

In [ ]:
from pathlib import Path
import scanpy as sc
import pandas as pd
import os


base_dir = os.environ.get("XENIUM_DATA_DIR", "path/to/data")


cell_matrices = {}
cell_metadata = {}

for patient_dir in sorted(base_dir.iterdir()):
    if not patient_dir.is_dir():
        continue

    # Find the files
    h5_files = list(patient_dir.glob("cell_feature_matrix.h5"))
    csv_files = list(patient_dir.glob("cells.csv"))

    if len(h5_files) != 1:
        print(f"{patient_dir.name}: expected 1 *_cell_feature_matrix.h5, found {len(h5_files)}")
        continue

    if len(csv_files) != 1:
        print(f"{patient_dir.name}: expected 1 *_cells.csv, found {len(csv_files)}")
        continue

    print(f"Loading {patient_dir.name}")

    cell_matrices[patient_dir.name] = sc.read_10x_h5(h5_files[0])
    cell_metadata[patient_dir.name] = pd.read_csv(csv_files[0])

print(f"\nLoaded {len(cell_matrices)} patients.")
print("Patients:", sorted(cell_matrices.keys()))

In [4]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from matplotlib.backends.backend_pdf import PdfPages

output_pdf = Path(base_dir) / "cell_matrices.pdf"
processed = []
skipped = []

required_cols = {"x_centroid", "y_centroid", "cell_area", "nucleus_area"}

with PdfPages(output_pdf) as pdf:
    for patient in sorted(cell_matrices.keys()):
        adata = cell_matrices[patient]
        meta = cell_metadata.get(patient)

        if meta is None:
            skipped.append((patient, "missing metadata"))
            continue

        meta = meta.copy()
        if len(meta) != adata.n_obs:
            skipped.append((patient, f"metadata rows ({len(meta)}) != cells ({adata.n_obs})"))
            continue

        meta.index = adata.obs_names
        adata.obs = meta.copy()

        missing_cols = sorted(required_cols - set(adata.obs.columns))
        if missing_cols:
            skipped.append((patient, f"missing columns: {', '.join(missing_cols)}"))
            continue

        adata.obsm["spatial"] = adata.obs[["x_centroid", "y_centroid"]].copy().to_numpy()
        sc.pp.calculate_qc_metrics(adata, percent_top=(10, 20, 50, 150), inplace=True)

        fig, axs = plt.subplots(1, 4, figsize=(15, 4))
        fig.suptitle(patient, y=1.05)

        axs[0].set_title("Total transcripts per cell")
        sns.histplot(adata.obs["total_counts"], kde=False, ax=axs[0])

        axs[1].set_title("Unique transcripts per cell")
        sns.histplot(adata.obs["n_genes_by_counts"], kde=False, ax=axs[1])

        axs[2].set_title("Area of segmented cells")
        sns.histplot(adata.obs["cell_area"], kde=False, ax=axs[2])

        axs[3].set_title("Nucleus ratio")
        nucleus_ratio = (adata.obs["nucleus_area"] / adata.obs["cell_area"]).replace([np.inf, -np.inf], np.nan)
        sns.histplot(nucleus_ratio.dropna(), kde=False, ax=axs[3])

        fig.tight_layout()
        pdf.savefig(fig, bbox_inches="tight")
        plt.close(fig)
        processed.append(patient)

print(f"Saved plots for {len(processed)} patients to {output_pdf}")
if skipped:
    print("Skipped patients:")
    for patient, reason in skipped:
        print(f"- {patient}: {reason}")

Saved plots for 4 patients to /home/gruengroup/jennifer/human/revision/xenium_2024/cell_matrices.pdf


In [ ]:
from pathlib import Path
import os

cwd = Path.cwd()
pdf_name = output_pdf if 'output_pdf' in globals() else 'cell_matrices.pdf'
pdf_path = (cwd / pdf_name).resolve()

print('Kernel CWD:', cwd)
print('Expected PDF path:', pdf_path)
print('Exists:', pdf_path.exists())

if pdf_path.exists():
    st = pdf_path.stat()
    print('Size bytes:', st.st_size)
    print('Modified:', st.st_mtime)

print('CWD files containing "cell_matrices":')
for p in sorted(cwd.glob('*cell_matrices*')):
    print('-', p)

print('os.getcwd():', os.getcwd())

In [ ]:
summary_rows = []

for patient in sorted(cell_matrices.keys()):
    adata = cell_matrices[patient]
    obs = adata.obs.copy()

    if "nucleus_ratio" not in obs.columns and {"nucleus_area", "cell_area"}.issubset(obs.columns):
        obs["nucleus_ratio"] = (obs["nucleus_area"] / obs["cell_area"]).replace([np.inf, -np.inf], np.nan)

    row = {
        "patient": patient,
        "n_cells": int(adata.n_obs),
        "median_total_counts": float(obs["total_counts"].median()) if "total_counts" in obs.columns else np.nan,
        "median_n_genes_by_counts": float(obs["n_genes_by_counts"].median()) if "n_genes_by_counts" in obs.columns else np.nan,
        "median_cell_area": float(obs["cell_area"].median()) if "cell_area" in obs.columns else np.nan,
        "median_nucleus_ratio": float(obs["nucleus_ratio"].median()) if "nucleus_ratio" in obs.columns else np.nan,
    }
    summary_rows.append(row)

summary_table = pd.DataFrame(summary_rows).sort_values("patient").reset_index(drop=True)
excel_path = Path(base_dir) / "cell_matrices_summary.xlsx"
summary_table.to_excel(excel_path, index=False)
print(f"Saved summary table to {excel_path}")
display(summary_table)